In [13]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.metrics import brier_score_loss


# ============================================================
# LOAD DATA
# ============================================================

matches = pd.read_csv('../data/processed/features_v6.csv')

# Make sure matches are in chronological order
matches['MatchDateTime'] = pd.to_datetime(matches['MatchDateTime'])
matches = matches.sort_values('MatchDateTime').reset_index(drop=True)


# ============================================================
# FEATURE SETS
# ============================================================

# Your current baseline
baseline_features = [
    'EloDiff',
    'ShotOTDiffLast5',
    'GoalAgainstDiffLast5'
]

# Overall/season xG
overall_xg_features = [
    'XGForDiffPg',
    'XGAgainstDiffPg',
    'XGDDiff'
]

# Recent xG
recent_xg_features = [
    'XGForDiffLast5',
    'XGAgainstDiffLast5',
    'XGDDiffLast5'
]

# Baseline + overall xG
baseline_overall_xg = (
    baseline_features +
    overall_xg_features
)

# Baseline + recent xG
baseline_recent_xg = (
    baseline_features +
    recent_xg_features
)

# Baseline + all xG
baseline_all_xg = (
    baseline_features +
    overall_xg_features +
    recent_xg_features
)


# ============================================================
# FUNCTION TO TEST A MODEL
# ============================================================
def test_model(features, name):

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    # Train on everything before 25-26
    train = matches[matches['Season'] != '25-26'].copy()

    # Test on 25-26
    test = matches[matches['Season'] == '25-26'].copy()

    # Number of games before removing missing values
    original_train_games = len(train)
    original_test_games = len(test)

    # Remove rows where required features are missing
    train = train.dropna(subset=features)
    test = test.dropna(subset=features)

    print(f"Features: {features}")

    print(f"\nOriginal training games: {original_train_games}")
    print(f"Usable training games:   {len(train)}")

    print(f"Original test games:     {original_test_games}")
    print(f"Usable test games:       {len(test)}")

    print("\nTest result distribution:")
    print(test['FTR'].value_counts())

    # Need at least two classes to calculate multiclass metrics
    if test['FTR'].nunique() < 2:

        print("\nWARNING: Test set contains fewer than 2 result classes.")
        print("Skipping model evaluation.")

        return {
            'name': name,
            'features': features,
            'model': None,
            'accuracy': np.nan,
            'log_loss': np.nan,
            'train_games': len(train),
            'test_games': len(test),
            'preds': None,
            'probs': None,
            'test': test,
            'importance': None
        }

    X_train = train[features]
    y_train = train['FTR']

    X_test = test[features]
    y_test = test['FTR']

    # Model
    model = LogisticRegression(
        max_iter=1000
    )

    model.fit(X_train, y_train)

    # Predictions
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, preds)

    logloss = log_loss(
        y_test,
        probs,
        labels=model.classes_
    )

    print(f"\nAccuracy: {accuracy:.4f}")
    print(f"Log Loss: {logloss:.4f}")

    # Brier scores
    classes = model.classes_

    for i, class_name in enumerate(classes):

        actual = (y_test == class_name).astype(int)

        brier = brier_score_loss(
            actual,
            probs[:, i]
        )

        print(f"{class_name} Brier Score: {brier:.4f}")

    # Coefficients
    importance = pd.DataFrame(
        model.coef_,
        columns=features,
        index=model.classes_
    )

    print("\nCoefficients:")
    print(importance)

    return {
        'name': name,
        'features': features,
        'model': model,
        'accuracy': accuracy,
        'log_loss': logloss,
        'train_games': len(train),
        'test_games': len(test),
        'preds': preds,
        'probs': probs,
        'test': test,
        'importance': importance
    }   

    

# ============================================================
# RUN THE TESTS
# ============================================================

results_baseline = test_model(
    baseline_features,
    'BASELINE'
)

results_overall_xg = test_model(
    baseline_overall_xg,
    'BASELINE + OVERALL XG'
)

results_recent_xg = test_model(
    baseline_recent_xg,
    'BASELINE + RECENT XG'
)

results_all_xg = test_model(
    baseline_all_xg,
    'BASELINE + ALL XG'
)


# ============================================================
# COMPARISON TABLE
# ============================================================

comparison = pd.DataFrame([
    {
        'Model': results_baseline['name'],
        'Features': len(results_baseline['features']),
        'TrainGames': results_baseline['train_games'],
        'TestGames': results_baseline['test_games'],
        'Accuracy': results_baseline['accuracy'],
        'LogLoss': results_baseline['log_loss']
    },
    {
        'Model': results_overall_xg['name'],
        'Features': len(results_overall_xg['features']),
        'TrainGames': results_overall_xg['train_games'],
        'TestGames': results_overall_xg['test_games'],
        'Accuracy': results_overall_xg['accuracy'],
        'LogLoss': results_overall_xg['log_loss']
    },
    {
        'Model': results_recent_xg['name'],
        'Features': len(results_recent_xg['features']),
        'TrainGames': results_recent_xg['train_games'],
        'TestGames': results_recent_xg['test_games'],
        'Accuracy': results_recent_xg['accuracy'],
        'LogLoss': results_recent_xg['log_loss']
    },
    {
        'Model': results_all_xg['name'],
        'Features': len(results_all_xg['features']),
        'TrainGames': results_all_xg['train_games'],
        'TestGames': results_all_xg['test_games'],
        'Accuracy': results_all_xg['accuracy'],
        'LogLoss': results_all_xg['log_loss']
    }
])

print("\n")
print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    comparison.sort_values(
        'LogLoss'
    ).to_string(index=False)
)


BASELINE
Features: ['EloDiff', 'ShotOTDiffLast5', 'GoalAgainstDiffLast5']

Original training games: 1520
Usable training games:   1520
Original test games:     380
Usable test games:       380

Test result distribution:
FTR
H    162
A    114
D    104
Name: count, dtype: int64

Accuracy: 0.4921
Log Loss: 1.0352
A Brier Score: 0.1966
D Brier Score: 0.2028
H Brier Score: 0.2249

Coefficients:
    EloDiff  ShotOTDiffLast5  GoalAgainstDiffLast5
A -0.003973        -0.044307             -0.071240
D  0.000595        -0.038053             -0.048897
H  0.003378         0.082360              0.120137

BASELINE + OVERALL XG
Features: ['EloDiff', 'ShotOTDiffLast5', 'GoalAgainstDiffLast5', 'XGForDiffPg', 'XGAgainstDiffPg', 'XGDDiff']

Original training games: 1520
Usable training games:   1480
Original test games:     380
Usable test games:       370

Test result distribution:
FTR
H    157
A    112
D    101
Name: count, dtype: int64

Accuracy: 0.4838
Log Loss: 1.0318
A Brier Score: 0.1979
D Brier S

In [14]:
import pandas as pd
import numpy as np

from itertools import combinations

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss
)


# ============================================================
# LOAD DATA
# ============================================================

matches = pd.read_csv(
    "../data/processed/features_v4.csv"
)

matches["MatchDateTime"] = pd.to_datetime(
    matches["MatchDateTime"]
)

matches = (
    matches
    .sort_values("MatchDateTime")
    .reset_index(drop=True)
)


# ============================================================
# BASELINE FEATURES
# ============================================================

baseline_features = [
    "EloDiff",
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5"
]


# ============================================================
# XG FEATURES TO ABLATE
# ============================================================

xg_features = [
    "XGForDiffPg",
    "XGAgainstDiffPg",
    "XGDDiff",

    "XGForDiffLast5",
    "XGAgainstDiffLast5",
    "XGDDiffLast5"
]


# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

train_original = matches[
    matches["Season"] != "25-26"
].copy()

test_original = matches[
    matches["Season"] == "25-26"
].copy()


print("=" * 80)
print("DATA")
print("=" * 80)

print("Training games:", len(train_original))
print("Test games:", len(test_original))

print("\nTest result distribution:")
print(test_original["FTR"].value_counts())


# ============================================================
# MODEL TEST FUNCTION
# ============================================================

def test_model(features, name):

    train = train_original.copy()
    test = test_original.copy()

    # --------------------------------------------------------
    # Only remove rows missing required features
    # --------------------------------------------------------

    train = train.dropna(subset=features)
    test = test.dropna(subset=features)

    # --------------------------------------------------------
    # Check that all three result classes exist
    # --------------------------------------------------------

    if test["FTR"].nunique() < 2:

        return {
            "Model": name,
            "Features": len(features),
            "FeatureList": " + ".join(features),
            "TrainGames": len(train),
            "TestGames": len(test),
            "Accuracy": np.nan,
            "LogLoss": np.nan,
            "Brier_A": np.nan,
            "Brier_D": np.nan,
            "Brier_H": np.nan
        }

    # --------------------------------------------------------
    # X / y
    # --------------------------------------------------------

    X_train = train[features]
    y_train = train["FTR"]

    X_test = test[features]
    y_test = test["FTR"]

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    model = LogisticRegression(
        max_iter=1000
    )

    model.fit(
        X_train,
        y_train
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        preds
    )

    logloss = log_loss(
        y_test,
        probs,
        labels=model.classes_
    )

    # --------------------------------------------------------
    # Brier scores
    # --------------------------------------------------------

    brier_scores = {}

    for i, class_name in enumerate(model.classes_):

        actual = (
            y_test == class_name
        ).astype(int)

        brier_scores[class_name] = (
            brier_score_loss(
                actual,
                probs[:, i]
            )
        )

    return {
        "Model": name,
        "Features": len(features),
        "FeatureList": " + ".join(features),
        "TrainGames": len(train),
        "TestGames": len(test),
        "Accuracy": accuracy,
        "LogLoss": logloss,
        "Brier_A": brier_scores.get("A", np.nan),
        "Brier_D": brier_scores.get("D", np.nan),
        "Brier_H": brier_scores.get("H", np.nan),
    }


# ============================================================
# BASELINE
# ============================================================

print()
print("=" * 80)
print("RUNNING BASELINE")
print("=" * 80)

baseline_result = test_model(
    baseline_features,
    "BASELINE"
)


# ============================================================
# SINGLE XG FEATURE ABLATION
# ============================================================

single_results = []

for feature in xg_features:

    features = (
        baseline_features +
        [feature]
    )

    result = test_model(
        features,
        f"BASELINE + {feature}"
    )

    single_results.append(result)


# ============================================================
# XG PAIR ABLATION
# ============================================================

pair_results = []

for combo in combinations(xg_features, 2):

    features = (
        baseline_features +
        list(combo)
    )

    result = test_model(
        features,
        "BASELINE + " + " + ".join(combo)
    )

    pair_results.append(result)


# ============================================================
# XG TRIPLE ABLATION
# ============================================================

triple_results = []

for combo in combinations(xg_features, 3):

    features = (
        baseline_features +
        list(combo)
    )

    result = test_model(
        features,
        "BASELINE + " + " + ".join(combo)
    )

    triple_results.append(result)


# ============================================================
# ALL XG
# ============================================================

all_xg_result = test_model(
    baseline_features + xg_features,
    "BASELINE + ALL XG"
)


# ============================================================
# COMBINE RESULTS
# ============================================================

all_results = (
    [baseline_result]
    + single_results
    + pair_results
    + triple_results
    + [all_xg_result]
)

results_df = pd.DataFrame(
    all_results
)


# ============================================================
# CALCULATE CHANGE FROM BASELINE
# ============================================================

baseline_accuracy = (
    baseline_result["Accuracy"]
)

baseline_logloss = (
    baseline_result["LogLoss"]
)

baseline_brier_a = (
    baseline_result["Brier_A"]
)

baseline_brier_d = (
    baseline_result["Brier_D"]
)

baseline_brier_h = (
    baseline_result["Brier_H"]
)


results_df["AccuracyChange"] = (
    results_df["Accuracy"]
    - baseline_accuracy
)

results_df["LogLossChange"] = (
    results_df["LogLoss"]
    - baseline_logloss
)

results_df["BrierAChange"] = (
    results_df["Brier_A"]
    - baseline_brier_a
)

results_df["BrierDChange"] = (
    results_df["Brier_D"]
    - baseline_brier_d
)

results_df["BrierHChange"] = (
    results_df["Brier_H"]
    - baseline_brier_h
)


# ============================================================
# SORT BY LOG LOSS
# LOWER = BETTER
# ============================================================

results_sorted = (
    results_df
    .sort_values(
        "LogLoss",
        ascending=True
    )
    .reset_index(drop=True)
)


# ============================================================
# PRINT MAIN RESULTS
# ============================================================

print()
print("=" * 100)
print("XG ABLATION RESULTS")
print("=" * 100)

print(
    results_sorted[
        [
            "Model",
            "Features",
            "TrainGames",
            "TestGames",
            "Accuracy",
            "LogLoss",
            "Brier_A",
            "Brier_D",
            "Brier_H",
            "AccuracyChange",
            "LogLossChange"
        ]
    ].to_string(index=False)
)


# ============================================================
# SINGLE FEATURE RESULTS
# ============================================================

single_df = pd.DataFrame(
    [baseline_result] + single_results
)

single_df["AccuracyChange"] = (
    single_df["Accuracy"]
    - baseline_accuracy
)

single_df["LogLossChange"] = (
    single_df["LogLoss"]
    - baseline_logloss
)

single_df = single_df.sort_values(
    "LogLoss"
)


print()
print("=" * 100)
print("SINGLE XG FEATURE RESULTS")
print("=" * 100)

print(
    single_df[
        [
            "Model",
            "Accuracy",
            "LogLoss",
            "AccuracyChange",
            "LogLossChange",
            "Brier_A",
            "Brier_D",
            "Brier_H"
        ]
    ].to_string(index=False)
)


# ============================================================
# BEST MODELS
# ============================================================

print()
print("=" * 100)
print("BEST MODELS")
print("=" * 100)


best_accuracy = results_df.loc[
    results_df["Accuracy"].idxmax()
]

best_logloss = results_df.loc[
    results_df["LogLoss"].idxmin()
]


print("\nBest Accuracy:")
print(
    best_accuracy[
        [
            "Model",
            "Accuracy",
            "LogLoss",
            "FeatureList"
        ]
    ]
)


print("\nBest Log Loss:")
print(
    best_logloss[
        [
            "Model",
            "Accuracy",
            "LogLoss",
            "FeatureList"
        ]
    ]
)


# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "../data/processed/xg_ablation_results.csv",
    index=False
)

print()
print("Saved:")
print("../data/processed/xg_ablation_results.csv")

DATA
Training games: 1520
Test games: 380

Test result distribution:
FTR
H    162
A    114
D    104
Name: count, dtype: int64

RUNNING BASELINE


KeyError: ['XGForDiffPg']